In [ ]:
!pip install SoccerNet

In [ ]:
"""
from SoccerNet.Downloader import SoccerNetDownloader
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory="/content")
mySoccerNetDownloader.downloadGames(files=["Labels-v3.json", "Frames-v3.zip"], split=["train","valid","test"], task="frames") # download frames and labels for the 400 games of SN v3 - Requires around 60 GB of local storage
"""

In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

In [ ]:
"""
with open("/content/sample_game/england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/Labels-v3.json") as f:
    data = json.load(f)
for img_name in list(data["actions"].keys())[:5]:  # first 5 images
    img_path = f"/content/sample_game/{img_name}"
    print(img_path)
    img = cv2.imread(img_path)

    if img is None:
        continue

    content = data["actions"][img_name]
    meta = content["imageMetadata"]
    label = meta["label"]
    game_time = meta["gameTime"]
    text = f"{label} | {game_time}"
    # Background rectangle (for readability)
    cv2.rectangle(img, (20, 20), (500, 80), (0, 0, 0), -1)
    # Text on top
    cv2.putText(img, text, (30, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            1, (255, 255, 255), 2)
    for box in content.get("bboxes", []):
        pts = box["points"]
        x1, y1 = int(pts["x1"]), int(pts["y1"])
        x2, y2 = int(pts["x2"]), int(pts["y2"])
        color = get_color(box["class"])

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

    cv2_imshow(img)
"""

In [ ]:
"""
Minimal Working Example — SoccerNet Action Spotting
=====================================================
Runs entirely on synthetic data (no download needed).
Demonstrates the full pipeline:
  1. Synthetic dataset that mimics SoccerNet features + labels
  2. Sliding-window model
  3. Training loop with class-weighted loss
  4. NMS post-processing
  5. Simple mAP evaluation

Requirements:
    pip install torch numpy scikit-learn
"""

In [ ]:
from SoccerNet.utils import getListGames
from SoccerNet.Downloader import SoccerNetDownloader
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
#Initialize downloader
downloader = SoccerNetDownloader(LocalDirectory="/content/")

downloader.password = "s0cc3rn3t"

files_to_download = ["Labels-v2.json", "1_224p.mkv","1_ResNET_TF2_PCA512.npy", "2_ResNET_TF2_PCA512.npy"]
split=["train", "valid", "test"]

subset = getListGames(split="train")[:45]
for i,game in enumerate(subset):
    print(f"Downloading {i}{game}...")
    downloader.downloadGame(game=game, files=files_to_download)
subset = getListGames(split="valid")[:20]
for i,game in enumerate(subset):
    print(f"Downloading {game}...")
    downloader.downloadGame(game=game, files=files_to_download)
print("✅ Download complete!")

In [ ]:
# Load labels for one of the downloaded games
def load_labels(local_dir, game):
    for root, dirs, files in os.walk(os.path.join(local_dir, game)):
        for f in files:
            if f.lower().endswith("labels-v2.json"):
                with open(os.path.join(root, f)) as fp:
                    return json.load(fp)
    return None

labels = load_labels("/content", subset[0])
if labels:
    print(f"Loaded labels")
else:
    print("No labels found for the selected game.")

In [ ]:
def load_real_game(feature_path, label_path):
    features = np.load(feature_path)

    with open(label_path) as f:
        labels_json = json.load(f)

    labels = np.zeros(len(features), dtype=np.int64)

    for ann in labels_json["annotations"]:
        frame = int(int(ann["position"]) / 500)

        if ann["label"] not in ACTION_NAME_TO_IDX:
            continue

        if frame < len(labels):
            labels[frame] = ACTION_NAME_TO_IDX[ann["label"]]

    return features, labels

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
FEATURE_DIM   = 512      # ResNet PCA feature size (matches real SoccerNet)
NUM_CLASSES   = 4        # simplified: Goal, Card, Substitution, Background
WINDOW_SIZE   = 15       # frames of temporal context (±7 around centre)
NUM_FRAMES    = 500      # frames per "game half" in our synthetic data
NUM_GAMES     = 8        # synthetic games
BATCH_SIZE    = 32
EPOCHS        = 10
LR            = 1e-3
NMS_WINDOW    = 10       # suppress detections within ±10 frames after a peak
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

ACTION_NAMES = [
    "Background",        # index 0 — no action
    "Penalty",
    "Kick-off",
    "Goal",
    "Substitution",
    "Offside",
    "Shots on target",
    "Shots off target",
    "Clearance",
    "Ball out of play",
    "Throw-in",
    "Foul",
    "Indirect free-kick",
    "Direct free-kick",
    "Corner",
    "Card",
]
# Map label strings → class indices (background not in JSON, handled separately)
ACTION_NAME_TO_IDX = {
    name: idx for idx, name in enumerate(ACTION_NAMES) if name != "Background"
}

NUM_CLASSES = len(ACTION_NAMES)
card_classes = [
    "Yellow card",
    "Red card",
    "Yellow->red card",
]



In [ ]:
class SoccerNetDataset(Dataset):
    def __init__(self, data_dir: str, split: str):
        self.data_dir = data_dir
        self.index = []  # list of (feature_path, label_path, frame_idx, label)

        games = getListGames(split=split)
        half_size = WINDOW_SIZE // 2

        for game in games:
            game_dir = os.path.join(data_dir, game)
            label_path = os.path.join(game_dir, "Labels-v2.json")

            for half_idx in [1, 2]:
                feature_path = os.path.join(game_dir, f"{half_idx}_ResNET_TF2_PCA512.npy")
                if not os.path.exists(feature_path) or not os.path.exists(label_path):
                    continue

                # Load labels only (tiny), not features
                _, labels = load_real_game(feature_path, label_path)
                T = len(labels)

                for t in range(T):
                    self.index.append((feature_path, t, T, labels[t]))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        feature_path, t, T, label = self.index[idx]
        half_size = WINDOW_SIZE // 2

        # Load only the relevant slice from disk
        features = np.load(feature_path, mmap_mode='r')  # memory-mapped!

        # Handle boundary padding manually
        start = t - half_size
        end   = t + half_size + 1
        pad_before = max(0, -start)
        pad_after  = max(0, end - T)
        start = max(0, start)
        end   = min(T, end)

        window = features[start:end]
        if pad_before or pad_after:
            window = np.pad(window, ((pad_before, pad_after), (0, 0)), mode='constant')

        return (
            torch.from_numpy(window.copy()),  # (WINDOW_SIZE, FEATURE_DIM)
            torch.tensor(label)
        )

In [ ]:
class ActionSpotter(nn.Module):
    def __init__(self, feature_dim=FEATURE_DIM, hidden_dim=256, num_classes=NUM_CLASSES):
        super().__init__()
        self.gru = nn.GRU(
            feature_dim, hidden_dim,
            batch_first=True, bidirectional=True
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):            # x: (B, W, D)
        out, _ = self.gru(x)         # (B, W, 2*hidden)
        # since your window is centered on frame t (t - W/2 ... t + W/2),
        # the center timestep is the most relevant one to classify
        center = out[:, out.size(1) // 2, :]
        return self.head(center)

In [ ]:
"""model = resnet18(weights=ResNet18_Weights.DEFAULT)
n_features = model.fc.in_features
print(n_features)
model.fc = nn.Sequential(nn.Linear(n_features, NUM_CLASSES))
print(model)
"""

In [ ]:
# ─────────────────────────────────────────────
# 3. CLASS-WEIGHTED LOSS (handles imbalance)
# ─────────────────────────────────────────────
def compute_class_weights(dataset: SoccerNetDataset) -> torch.Tensor:
    labels = np.array([entry[3] for entry in dataset.index])  # extract label from each tuple
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    counts = np.where(counts == 0, 1, counts)
    weights = 1.0 / counts
    weights = weights / weights.sum() * NUM_CLASSES
    return torch.tensor(weights, dtype=torch.float32)

In [ ]:
# ─────────────────────────────────────────────
# 4. NON-MAXIMUM SUPPRESSION
# ─────────────────────────────────────────────
def nms(scores: np.ndarray, window: int = NMS_WINDOW) -> np.ndarray:
    """
    scores : (T, NUM_CLASSES) — softmax probabilities, background excluded
    Returns a sparse (T, NUM_CLASSES-1) array with non-peak scores zeroed out.
    """
    T, C = scores.shape
    suppressed = scores.copy()
    for c in range(C):
        col = scores[:, c]
        for t in range(T):
            if col[t] == 0:
                continue
            lo, hi = max(0, t - window), min(T, t + window + 1)
            if col[t] < col[lo:hi].max():
                suppressed[t, c] = 0.0   # suppress non-peak
    return suppressed

In [ ]:
# ─────────────────────────────────────────────
# 5. TRAINING
# ─────────────────────────────────────────────
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

In [ ]:
# ─────────────────────────────────────────────
# 6. EVALUATION  (simple mAP per action class)
# ─────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            logits = model(X.to(DEVICE))
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y.numpy())

    probs  = np.concatenate(all_probs)    # (N, NUM_CLASSES)
    labels = np.concatenate(all_labels)   # (N,)

    # Apply NMS on the action columns (skip background col 0)
    action_probs_nms = nms(probs[:, 1:])  # (N, NUM_CLASSES-1)

    ap_scores = []
    for c in range(1, NUM_CLASSES):   # skip index 0 (Background)
        binary_labels = (labels == c).astype(int)
        if binary_labels.sum() == 0:
            continue
        ap = average_precision_score(binary_labels, action_probs_nms[:, c - 1])
        ap_scores.append(ap)
        print(f"  AP [{ACTION_NAMES[c]:>20s}]: {ap:.4f}")

    mean_ap = np.mean(ap_scores) if ap_scores else 0.0
    print(f"  {'mAP':>16s} : {mean_ap:.4f}")
    return mean_ap


In [ ]:
# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn
def main():
    print("=" * 50)
    print(" SoccerNet Action Spotting — Real Data")
    print(f" Device : {DEVICE}")
    print("=" * 50)

    DATA_DIR = "/content"   # ← change this if your data lives elsewhere

    # --- Data ---
    print("\n[1/3] Loading real SoccerNet data...")
    train_set = SoccerNetDataset(data_dir=DATA_DIR, split="train")
    val_set   = SoccerNetDataset(data_dir=DATA_DIR, split="valid")

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True, drop_last=False,num_workers=2, pin_memory=True)
    val_loader   = DataLoader(train_set, batch_size=32, shuffle=True, drop_last=False, num_workers=2, pin_memory=True)

    train_labels = np.array([entry[3] for entry in train_set.index])
    label_counts = np.bincount(train_labels, minlength=NUM_CLASSES)

    print(f"  Train samples : {len(train_set)}")
    print(f"  Val   samples : {len(val_set)}")
    print(f"  Label dist (train):")
    for i, name in enumerate(ACTION_NAMES):
        print(f"    {name:>20s}: {int(label_counts[i])}")

    # --- Model ---
    print("\n[2/3] Training...")
    model     = ActionSpotter().to(DEVICE)
    print(model)
    weights   = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        loss = train(model, train_loader, optimizer, criterion)
        if epoch % 2 == 0 or epoch == 1:
            print(f"  Epoch {epoch:02d}/{EPOCHS}  loss={loss:.4f}")

    # --- Evaluate ---
    print("\n[3/3] Evaluation on validation set (with NMS):")
    evaluate(model, val_loader)


if __name__ == "__main__":
    main()

In [ ]:

grpu_train_35_10="""==================================================
 SoccerNet Action Spotting — Real Data
 Device : cuda
==================================================

[1/3] Loading real SoccerNet data...
  Train samples : 388860
  Val   samples : 220726
  Label dist (train):
              Background: 374724
                 Penalty: 8
                Kick-off: 265
                    Goal: 195
            Substitution: 360
                 Offside: 215
         Shots on target: 782
        Shots off target: 643
               Clearance: 1100
        Ball out of play: 4575
                Throw-in: 2687
                    Foul: 1360
      Indirect free-kick: 953
        Direct free-kick: 274
                  Corner: 719
                    Card: 0

[2/3] Training...
ActionSpotter(
  (gru): GRU(512, 256, batch_first=True, bidirectional=True)
  (head): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=16, bias=True)
  )
)
  Epoch 01/10  loss=2.0236
  Epoch 02/10  loss=1.8974
  Epoch 04/10  loss=1.8014
  Epoch 06/10  loss=1.7503
  Epoch 08/10  loss=1.7072
  Epoch 10/10  loss=1.7361

[3/3] Evaluation on validation set (with NMS):
  AP [             Penalty]: 0.0001
  AP [            Kick-off]: 0.3224
  AP [                Goal]: 0.1133
  AP [        Substitution]: 0.1518
  AP [             Offside]: 0.0146
  AP [     Shots on target]: 0.0728
  AP [    Shots off target]: 0.1131
  AP [           Clearance]: 0.1342
  AP [    Ball out of play]: 0.0688
  AP [            Throw-in]: 0.0844
  AP [                Foul]: 0.0742
  AP [  Indirect free-kick]: 0.0480
  AP [    Direct free-kick]: 0.0879
  AP [              Corner]: 0.2103
               mAP : 0.1069"""
last_gru="""
==================================================
 SoccerNet Action Spotting — Real Data
 Device : cuda
==================================================

[1/3] Loading real SoccerNet data...
  Train samples : 111645
  Val   samples : 110983
  Label dist (train):
              Background: 107538
                 Penalty: 2
                Kick-off: 71
                    Goal: 52
            Substitution: 108
                 Offside: 69
         Shots on target: 245
        Shots off target: 184
               Clearance: 316
        Ball out of play: 1288
                Throw-in: 759
                    Foul: 433
      Indirect free-kick: 266
        Direct free-kick: 61
                  Corner: 253
                    Card: 0

[2/3] Training...
ActionSpotter(
  (gru): GRU(512, 256, batch_first=True, bidirectional=True)
  (head): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=16, bias=True)
  )
)
  Epoch 01/10  loss=2.0889
  Epoch 02/10  loss=1.8931
  Epoch 04/10  loss=1.4819
  Epoch 06/10  loss=1.1457
  Epoch 08/10  loss=0.9326
  Epoch 10/10  loss=0.8144

[3/3] Evaluation on validation set (with NMS):
  AP [             Penalty]: 0.0026
  AP [            Kick-off]: 0.6261
  AP [                Goal]: 0.3576
  AP [        Substitution]: 0.4899
  AP [             Offside]: 0.4447
  AP [     Shots on target]: 0.4868
  AP [    Shots off target]: 0.5068
  AP [           Clearance]: 0.4323
  AP [    Ball out of play]: 0.1806
  AP [            Throw-in]: 0.2850
  AP [                Foul]: 0.3093
  AP [  Indirect free-kick]: 0.4434
  AP [    Direct free-kick]: 0.4392
  AP [              Corner]: 0.4670
               mAP : 0.3908"""
last="""
[3/3] Evaluation on validation set (with NMS):
  AP [             Penalty]: 0.0000
  AP [            Kick-off]: 0.1853
  AP [                Goal]: 0.0071
  AP [        Substitution]: 0.0914
  AP [             Offside]: 0.0007
  AP [     Shots on target]: 0.0273
  AP [    Shots off target]: 0.0878
  AP [           Clearance]: 0.1891
  AP [    Ball out of play]: 0.0969
  AP [            Throw-in]: 0.1195
  AP [                Foul]: 0.0364
  AP [  Indirect free-kick]: 0.0176
  AP [    Direct free-kick]: 0.0006
  AP [              Corner]: 0.2106
               mAP : 0.0764
"""
sa="""
[3/3] Evaluation on validation set (with NMS):
  AP [             Penalty]: 0.0001
  AP [            Kick-off]: 0.0006
  AP [                Goal]: 0.0003
  AP [        Substitution]: 0.0017
  AP [             Offside]: 0.0006
  AP [     Shots on target]: 0.0046
  AP [    Shots off target]: 0.0012
  AP [           Clearance]: 0.0520
  AP [    Ball out of play]: 0.0136
  AP [            Throw-in]: 0.0101
  AP [                Foul]: 0.0052
  AP [  Indirect free-kick]: 0.0021
  AP [    Direct free-kick]: 0.0007
  AP [              Corner]: 0.0010
               mAP : 0.0067"""

In [ ]:
!git clone https://github.com/SilvioGiancola/SoccerNetv2-DevKit.git /.

fatal: destination path '/.' already exists and is not an empty directory.


In [ ]:
!mkdir /content/SoccerNet & mv /content/england_epl/ /content/europe_uefa-champions-league/ /content/SoccerNet

In [ ]:
!rm /content/SoccerNet/ -r

In [ ]:
!cd /content/SoccerNetv2-DevKit/Task1-ActionSpotting/TemporallyAwarePooling && python src/main.py --SoccerNet_path=/content/SoccerNet/ --model_name=NetVLAD++

2026-07-06 14:27:45,660 [MainThread  ] [INFO ]  Starting main function
2026-07-06 14:27:45,660 [MainThread  ] [INFO ]  Parameters:
2026-07-06 14:27:45,660 [MainThread  ] [INFO ]   SoccerNet_path : /content/SoccerNet/
2026-07-06 14:27:45,660 [MainThread  ] [INFO ]         features : ResNET_TF2.npy
2026-07-06 14:27:45,660 [MainThread  ] [INFO ]       max_epochs : 1000
2026-07-06 14:27:45,660 [MainThread  ] [INFO ]     load_weights : None
2026-07-06 14:27:45,660 [MainThread  ] [INFO ]       model_name : NetVLAD++
2026-07-06 14:27:45,661 [MainThread  ] [INFO ]        test_only : False
2026-07-06 14:27:45,661 [MainThread  ] [INFO ]      split_train : ['train']
2026-07-06 14:27:45,661 [MainThread  ] [INFO ]      split_valid : ['valid']
2026-07-06 14:27:45,661 [MainThread  ] [INFO ]       split_test : ['test', 'challenge']
2026-07-06 14:27:45,661 [MainThread  ] [INFO ]          version : 2
2026-07-06 14:27:45,661 [MainThread  ] [INFO ]      feature_dim : None
2026-07-06 14:27:45,661 [MainThre